# GlobalPositioner

> **Created by Codex.**

Jointly estimate camera and landmark positions from camera-to-point directions using the GLOMAP-style BATA model.

Source: [`GlobalPositioner.h`](https://github.com/borglab/gtsam/blob/develop/gtsam/sfm/GlobalPositioner.h)

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/sfm/doc/GlobalPositioner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [ ]:
import gtsam
import numpy as np

from gtsam import symbol_shorthand

C = symbol_shorthand.C
K = symbol_shorthand.K
P = symbol_shorthand.P
S = symbol_shorthand.S
X = symbol_shorthand.X

## Problem structure

`GlobalPositioner` specializes `LocationRecovery` for a bipartite camera-landmark graph. It always uses bilinear BATA factors, validates that the anchor is a camera, initializes one scale per observation, and runs Levenberg-Marquardt.

The input directions must point from the camera key (`key1`) to the landmark key (`key2`) in a common world frame.

In [ ]:
direction_noise = gtsam.noiseModel.Isotropic.Sigma(2, 0.01)
camera_keys = {C(0), C(1)}
landmark_keys = {P(0), P(1)}
directions = [
    gtsam.BinaryMeasurementUnit3(C(0), P(0), gtsam.Unit3(np.array([0.0, 0.0, 1.0])), direction_noise),
    gtsam.BinaryMeasurementUnit3(C(0), P(1), gtsam.Unit3(np.array([1.0, 0.0, 1.0])), direction_noise),
    gtsam.BinaryMeasurementUnit3(C(1), P(0), gtsam.Unit3(np.array([-1.0, 0.0, 1.0])), direction_noise),
    gtsam.BinaryMeasurementUnit3(C(1), P(1), gtsam.Unit3(np.array([0.0, 0.0, 1.0])), direction_noise),
]

positioner = gtsam.GlobalPositioner()
initial = positioner.initializeRandomly(camera_keys, landmark_keys, directions)
graph = positioner.buildGraph(directions, bilinear=True)
positioner.addAnchorPrior(C(0), graph)

print("measurement factors plus anchor:", graph.size())
print("initial keys:", initial.keys())

## Running a real problem

Call `positioner.run(directions, camera_keys, landmark_keys, anchor_camera, initial)` once the graph is well connected and the initialization contains any available metric information. A direction-only problem retains a global scale ambiguity; in production, seed or constrain scale through the surrounding reconstruction pipeline and check rank if LM reports an indeterminate system.